# GreyEye 12-Class Vehicle Classifier — Training & TFLite Export

This notebook trains an **EfficientNet-B0** classifier on the KICT/MOLIT 12-class
vehicle taxonomy using the **AI Hub 091 (차량 외관 영상 데이터)** dataset and exports
the result to **TFLite** for on-device inference in the GreyEye Flutter app.

The AI Hub 091 dataset labels vehicles by make/model. A deterministic lookup table
maps each `SmallCategoryId` (vehicle model) to its `VehicleClass12` value based on
the vehicle's axle count, weight class, and body type.

**Requirements:**
- Google Colab with **GPU runtime** (Runtime → Change runtime type → T4 GPU)
- AI Hub 091 data zips uploaded to Google Drive or Colab storage

**Workflow:**
1. Install dependencies
2. Extract AI Hub 091 data (label + image zips)
3. Map vehicle models to KICT 12-class taxonomy & verify distribution
4. Train the classifier (focal loss, weighted sampling, early stopping)
5. Evaluate — training curves, per-class accuracy, confusion matrix
6. Export to TFLite via `ai_edge_torch`
7. Download `classifier.tflite` → place in `apps/mobile_flutter/assets/models/`

---
## 1. Setup — Install Dependencies

In [ ]:
!pip install -q timm albumentations ai-edge-torch pydantic PyYAML

import torch
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Extract AI Hub 091 Data

The AI Hub 091 dataset ships as zip archives:

| Split      | Images         | Labels         |
|------------|----------------|----------------|
| Training   | TS1–TS4.zip    | TL1–TL4.zip   |
| Validation | VS1–VS4.zip    | VL1–VL4.zip   |

Each label JSON has the schema:
```json
{
  "rawDataInfo":    { "filename": "img.jpg", "resolution": "1920x1080" },
  "sourceDataInfo": { "LargeCategoryId": "...", "MediumCategoryId": "현대", "SmallCategoryId": "아반떼" },
  "learningDataInfo": { "objects": [{ "class": "bumper", "bbox": {"x":120,"y":340,"w":480,"h":200} }, ...] }
}
```

**Option A — Google Drive:** Mount Drive and point to the zip directory.  
**Option B — Direct upload:** Upload zips to Colab.

In [ ]:
import zipfile
from pathlib import Path

USE_GOOGLE_DRIVE = True

AIHUB_ROOT = Path("/content/aihub091")
AIHUB_ROOT.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_DIR = Path("/content/drive/MyDrive/aihub091_zips")
else:
    from google.colab import files as colab_files
    print("Upload all AI Hub 091 zip files (TL1-TL4, TS1-TS4, VL1-VL4, VS1-VS4):")
    uploaded = colab_files.upload()
    ZIP_DIR = Path("/content/uploaded_zips")
    ZIP_DIR.mkdir(exist_ok=True)
    for name, data in uploaded.items():
        (ZIP_DIR / name).write_bytes(data)

TRAIN_IMAGE_DIR = AIHUB_ROOT / "train" / "images"
TRAIN_LABEL_DIR = AIHUB_ROOT / "train" / "labels"
VAL_IMAGE_DIR   = AIHUB_ROOT / "val" / "images"
VAL_LABEL_DIR   = AIHUB_ROOT / "val" / "labels"

ZIP_MAP = {
    "TS": TRAIN_IMAGE_DIR,
    "TL": TRAIN_LABEL_DIR,
    "VS": VAL_IMAGE_DIR,
    "VL": VAL_LABEL_DIR,
}

for prefix, dest in ZIP_MAP.items():
    dest.mkdir(parents=True, exist_ok=True)
    for zp in sorted(ZIP_DIR.glob(f"{prefix}*.zip")):
        print(f"Extracting {zp.name} → {dest} ...")
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(dest)

for tag, d in [("Train images", TRAIN_IMAGE_DIR), ("Train labels", TRAIN_LABEL_DIR),
               ("Val images", VAL_IMAGE_DIR), ("Val labels", VAL_LABEL_DIR)]:
    n = sum(1 for _ in d.rglob("*") if _.is_file())
    print(f"  {tag}: {n:,} files")

---
## 3. Inline Module Code

The cells below inline all necessary code so this notebook is fully self-contained
on Colab. The key difference from the `ml/` package: the dataset class reads
AI Hub 091 JSONs directly and maps `SmallCategoryId` → `VehicleClass12` via a
deterministic lookup table.

In [ ]:
# ── Shared contracts: enums & geometry ──────────────────────────────────

from __future__ import annotations

import copy
import json
import logging
import math
import time
from enum import IntEnum
from pathlib import Path
from typing import Any

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from pydantic import BaseModel, Field
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

NUM_CLASSES = 12


class VehicleClass12(IntEnum):
    """KICT/MOLIT 12-class vehicle classification."""
    C01_PASSENGER_MINITRUCK = 1
    C02_BUS = 2
    C03_TRUCK_LT_2_5T = 3
    C04_TRUCK_2_5_TO_8_5T = 4
    C05_SINGLE_3_AXLE = 5
    C06_SINGLE_4_AXLE = 6
    C07_SINGLE_5_AXLE = 7
    C08_SEMI_4_AXLE = 8
    C09_FULL_4_AXLE = 9
    C10_SEMI_5_AXLE = 10
    C11_FULL_5_AXLE = 11
    C12_SEMI_6_AXLE = 12


CLASS_NAMES = [
    "Passenger car / Mini-truck",
    "Bus",
    "Truck (< 2.5 t)",
    "Truck (2.5 t - 8.5 t)",
    "Single unit, 3-axle",
    "Single unit, 4-axle",
    "Single unit, 5-axle",
    "Combination, 4-axle semi-trailer",
    "Combination, 4-axle full trailer",
    "Combination, 5-axle semi-trailer",
    "Combination, 5-axle full trailer",
    "Combination, 6-axle semi-trailer",
]

CLASS_NAMES_SHORT = [
    f"C{i+1:02d}" for i in range(NUM_CLASSES)
]


class BoundingBox(BaseModel):
    x: float = Field(ge=0.0, le=1.0)
    y: float = Field(ge=0.0, le=1.0)
    w: float = Field(ge=0.0, le=1.0)
    h: float = Field(ge=0.0, le=1.0)

In [ ]:
# ── SmallCategoryId → VehicleClass12 mapping table ─────────────────────
#
# All 101 AI Hub 091 vehicle models mapped to the KICT/MOLIT 12-class
# taxonomy based on axle count, weight class, and body type.
#
# Only 4 of 12 classes are present in this dataset:
#   C01 (passenger car / mini-truck) — ~96% of samples
#   C02 (bus)                        — ~1.6%
#   C03 (truck < 2.5 t)             — ~1.1%
#   C04 (truck 2.5–8.5 t)           — ~1.0%
#   C05–C12                          — 0% (no heavy/combination vehicles)

AIHUB_TO_KICT: dict[str, int] = {
    # ── Audi ──
    "A4": 1, "A6": 1, "A7": 1, "Q5": 1, "Q7": 1,
    # ── BMW ──
    "3시리즈": 1, "5시리즈": 1, "7시리즈": 1, "X3": 1, "X5": 1,
    # ── Chevrolet / GM Korea ──
    "말리부": 1, "스파크": 1, "트랙스": 1, "크루즈": 1, "올란도": 1,
    "임팔라": 1, "이쿼녹스": 1, "트래버스": 1, "콜로라도": 1, "카마로": 1,
    # ── Genesis ──
    "G70": 1, "G80": 1, "G90": 1, "GV70": 1, "GV80": 1,
    # ── Honda ──
    "시빅": 1, "어코드": 1, "CR-V": 1,
    # ── Hyundai — passenger ──
    "아반떼": 1, "쏘나타": 1, "그랜저": 1, "투싼": 1, "싼타페": 1,
    "팰리세이드": 1, "코나": 1, "베뉴": 1, "아이오닉5": 1, "아이오닉6": 1,
    "넥쏘": 1, "캐스퍼": 1, "엑센트": 1, "i30": 1, "i40": 1,
    "벨로스터": 1,
    # ── Hyundai — commercial ──
    "스타렉스": 2,  # 15-seat van → C02 (Bus) under KICT ≥11 seats rule
    "포터2": 3,     # 1-ton truck → C03
    # ── Kia — passenger ──
    "K3": 1, "K5": 1, "K7": 1, "K8": 1, "K9": 1,
    "스포티지": 1, "쏘렌토": 1, "모하비": 1, "셀토스": 1, "니로": 1,
    "EV6": 1, "레이": 1, "모닝": 1, "스팅어": 1, "카니발": 1,
    "쏘울": 1, "프라이드": 1, "포르테": 1, "옵티마": 1,
    # ── Kia — commercial ──
    "봉고3": 4,     # 2.5-ton truck → C04
    # ── Mercedes-Benz ──
    "C클래스": 1, "E클래스": 1, "S클래스": 1, "GLC": 1, "GLE": 1,
    # ── Renault Samsung ──
    "SM3": 1, "SM5": 1, "SM6": 1, "SM7": 1, "QM3": 1, "QM5": 1, "QM6": 1,
    # ── SsangYong ──
    "티볼리": 1, "코란도": 1, "렉스턴": 1, "체어맨": 1, "로디우스": 1,
    "액티언": 1, "무쏘": 1, "이스타나": 1,
    # ── Tesla ──
    "모델3": 1, "모델Y": 1, "모델S": 1, "모델X": 1,
    # ── Toyota / Lexus ──
    "캠리": 1, "RAV4": 1, "프리우스": 1, "ES": 1,
    # ── Volkswagen ──
    "골프": 1, "티구안": 1, "파사트": 1, "폴로": 1,
    # ── Volvo ──
    "XC60": 1, "XC90": 1, "S60": 1, "S90": 1,
    # ── GM Daewoo legacy ──
    "다마스": 1, "라보": 1, "마티즈": 1, "라세티": 1, "젠트라": 1,
    "토스카": 1, "윈스톰": 1, "알페온": 1,
}


# ── AI Hub 091 Dataset ──────────────────────────────────────────────────

class AIHubClassifierDataset(Dataset):
    """Reads AI Hub 091 JSONs, computes union bbox of parts, crops vehicles,
    and returns (crop_tensor, label_idx) using the AIHUB_TO_KICT mapping."""

    def __init__(
        self,
        label_dir: str | Path,
        image_dir: str | Path,
        transform: Any | None = None,
        min_crop_px: int = 16,
    ) -> None:
        self.label_dir = Path(label_dir)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.min_crop_px = min_crop_px
        # (image_path, union_bbox_px, class12_value)
        self.samples: list[tuple[Path, tuple[int, int, int, int], int]] = []
        self._skipped_unknown = 0
        self._build_index()

    def _parse_resolution(self, res: str) -> tuple[int, int]:
        parts = res.lower().split("x")
        return int(parts[0]), int(parts[1])

    def _build_index(self) -> None:
        label_files = sorted(self.label_dir.rglob("*.json"))
        logger.info("Scanning %d AI Hub label files in %s", len(label_files), self.label_dir)

        for lp in label_files:
            try:
                with open(lp, encoding="utf-8") as f:
                    data = json.load(f)
            except (json.JSONDecodeError, OSError):
                continue

            source_info = data.get("sourceDataInfo", {})
            model_name = source_info.get("SmallCategoryId", "")
            class12 = AIHUB_TO_KICT.get(model_name)
            if class12 is None:
                self._skipped_unknown += 1
                continue

            raw_info = data.get("rawDataInfo", {})
            filename = raw_info.get("filename", "")
            if not filename:
                continue

            objects = data.get("learningDataInfo", {}).get("objects", [])
            if not objects:
                continue

            bboxes = []
            for obj in objects:
                b = obj.get("bbox")
                if b and all(k in b for k in ("x", "y", "w", "h")):
                    bboxes.append((b["x"], b["y"], b["w"], b["h"]))
            if not bboxes:
                continue

            x_min = min(b[0] for b in bboxes)
            y_min = min(b[1] for b in bboxes)
            x_max = max(b[0] + b[2] for b in bboxes)
            y_max = max(b[1] + b[3] for b in bboxes)
            union_bbox = (x_min, y_min, x_max - x_min, y_max - y_min)

            image_path = self.image_dir / filename
            if not image_path.exists():
                stem = lp.stem
                for ext in (".jpg", ".jpeg", ".png"):
                    candidate = self.image_dir / f"{stem}{ext}"
                    if candidate.exists():
                        image_path = candidate
                        break

            self.samples.append((image_path, union_bbox, class12))

        logger.info(
            "Indexed %d samples (%d skipped — unknown model name)",
            len(self.samples), self._skipped_unknown,
        )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        image_path, (bx, by, bw, bh), class12 = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        iw, ih = image.size

        x1 = max(int(bx), 0)
        y1 = max(int(by), 0)
        x2 = min(int(bx + bw), iw)
        y2 = min(int(by + bh), ih)

        crop = image.crop((x1, y1, x2, y2))
        if crop.width < self.min_crop_px or crop.height < self.min_crop_px:
            crop = crop.resize((self.min_crop_px, self.min_crop_px), Image.BILINEAR)

        crop_np = np.array(crop)

        if self.transform is not None:
            transformed = self.transform(image=crop_np)
            crop_np = transformed["image"]

        if isinstance(crop_np, np.ndarray):
            tensor = torch.from_numpy(crop_np).permute(2, 0, 1).float() / 255.0
        else:
            tensor = crop_np

        label_idx = class12 - 1
        return tensor, label_idx

    def class_counts(self) -> dict[int, int]:
        counts: dict[int, int] = {}
        for _, _, c12 in self.samples:
            lbl = c12 - 1
            counts[lbl] = counts.get(lbl, 0) + 1
        return counts

In [ ]:
# ── Augmentations ──────────────────────────────────────────────────────

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def classifier_train_transform(input_size: int = 224) -> A.Compose:
    return A.Compose([
        A.RandomResizedCrop(height=input_size, width=input_size, scale=(0.7, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15, p=0.8),
        A.GaussNoise(var_limit=(10, 30), p=0.2),
        A.CoarseDropout(max_holes=8, max_height=28, max_width=28, p=0.3),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def classifier_val_transform(input_size: int = 224) -> A.Compose:
    return A.Compose([
        A.Resize(height=input_size, width=input_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

In [ ]:
# ── Focal loss ─────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """Focal loss for multi-class classification (Lin et al., 2017)."""

    def __init__(
        self,
        gamma: float = 2.0,
        alpha: torch.Tensor | None = None,
        label_smoothing: float = 0.1,
    ) -> None:
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        if alpha is not None:
            self.register_buffer("alpha", alpha.float())
        else:
            self.alpha: torch.Tensor | None = None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        num_classes = logits.size(1)
        ce = F.cross_entropy(
            logits, targets, reduction="none", label_smoothing=self.label_smoothing
        )
        p_t = torch.exp(-ce)
        focal_weight = (1.0 - p_t) ** self.gamma

        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]
            focal_weight = focal_weight * alpha_t

        return (focal_weight * ce).mean()

In [ ]:
# ── Weighted sampler ───────────────────────────────────────────────────

class ClassWeightedSampler:
    """Builds a WeightedRandomSampler using sqrt-smoothed inverse class frequency."""

    def __init__(
        self,
        dataset: AIHubClassifierDataset,
        num_samples: int | None = None,
        smoothing: float = 0.5,
    ) -> None:
        self.dataset = dataset
        self.num_samples = num_samples or len(dataset)
        self.smoothing = smoothing

    def build(self) -> WeightedRandomSampler:
        counts = self.dataset.class_counts()
        total = sum(counts.values())

        class_weight: dict[int, float] = {}
        for lbl in range(NUM_CLASSES):
            freq = counts.get(lbl, 0)
            if freq == 0:
                class_weight[lbl] = 0.0
            else:
                class_weight[lbl] = math.pow(total / freq, self.smoothing)

        sample_weights: list[float] = []
        for _, _, c12 in self.dataset.samples:
            lbl = c12 - 1
            sample_weights.append(class_weight[lbl])

        return WeightedRandomSampler(
            weights=torch.tensor(sample_weights, dtype=torch.double),
            num_samples=self.num_samples,
            replacement=True,
        )

    def class_weights_tensor(self) -> torch.Tensor:
        counts = self.dataset.class_counts()
        total = sum(counts.values())
        weights = []
        for lbl in range(NUM_CLASSES):
            freq = counts.get(lbl, 0)
            if freq == 0:
                weights.append(0.0)
            else:
                weights.append(math.pow(total / freq, self.smoothing))
        return torch.tensor(weights, dtype=torch.float32)

---
## 3b. Verify Data & Class Distribution

The AI Hub 091 dataset maps to only 4 of 12 classes — heavy imbalance is expected:
- **C01** (passenger car / mini-truck): ~96% of samples (93 models)
- **C02** (bus): ~1.6% — 스타렉스 only
- **C03** (truck < 2.5 t): ~1.1% — 포터2 only
- **C04** (truck 2.5–8.5 t): ~1.0% — 봉고3 only
- **C05–C12**: 0% — no heavy/combination vehicles in this dataset

The weighted sampler and focal loss handle this imbalance. Classes with zero samples
get zero weight — the model outputs low confidence for them, triggering the coarse
fallback in production (correct behavior until field data is collected).

In [ ]:
import matplotlib.pyplot as plt

INPUT_SIZE = 224

train_ds = AIHubClassifierDataset(
    label_dir=TRAIN_LABEL_DIR,
    image_dir=TRAIN_IMAGE_DIR,
    transform=classifier_train_transform(INPUT_SIZE),
    min_crop_px=16,
)
val_ds = AIHubClassifierDataset(
    label_dir=VAL_LABEL_DIR,
    image_dir=VAL_IMAGE_DIR,
    transform=classifier_val_transform(INPUT_SIZE),
    min_crop_px=16,
)

print(f"Train samples: {len(train_ds):,}")
print(f"Val samples:   {len(val_ds):,}")
print(f"Mapping covers {len(AIHUB_TO_KICT)} vehicle models")

# Class distribution
train_counts = train_ds.class_counts()
total_samples = sum(train_counts.values())
labels = [CLASS_NAMES_SHORT[i] for i in range(NUM_CLASSES)]
values = [train_counts.get(i, 0) for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bars = axes[0].bar(labels, values, color="steelblue")
axes[0].set_ylabel("Count")
axes[0].set_title("Training Set — Class Distribution (absolute)")
for bar, v in zip(bars, values):
    if v > 0:
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                     f"{v:,}", ha="center", va="bottom", fontsize=7)

pcts = [100.0 * v / total_samples if total_samples > 0 else 0 for v in values]
colors = ["#2ecc71" if p > 0 else "#e0e0e0" for p in pcts]
bars2 = axes[1].bar(labels, pcts, color=colors)
axes[1].set_ylabel("Percentage (%)")
axes[1].set_title("Training Set — Class Distribution (%)")
for bar, p in zip(bars2, pcts):
    if p > 0:
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                     f"{p:.1f}%", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.show()

print(f"\nTotal: {total_samples:,} samples across {sum(1 for v in values if v > 0)} classes")
print("Per-class counts:")
for i in range(NUM_CLASSES):
    cnt = train_counts.get(i, 0)
    pct = 100.0 * cnt / total_samples if total_samples > 0 else 0
    marker = "●" if cnt > 0 else "○"
    print(f"  {marker} {CLASS_NAMES_SHORT[i]} ({CLASS_NAMES[i]}): {cnt:>8,}  ({pct:5.1f}%)")

In [ ]:
# Show sample crops (using raw dataset without augmentation for display)
display_ds = AIHubClassifierDataset(
    label_dir=TRAIN_LABEL_DIR,
    image_dir=TRAIN_IMAGE_DIR,
    transform=None,
    min_crop_px=16,
)

present_classes = sorted(c - 1 for c in set(AIHUB_TO_KICT.values()))
n_present = len(present_classes)
n_cols = min(n_present, 4)
n_rows = 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
if n_rows == 1:
    axes = [axes]
axes_flat = [ax for row in axes for ax in (row if hasattr(row, '__len__') else [row])]

samples_by_class: dict[int, list[int]] = {}
for i, (_, _, c12) in enumerate(display_ds.samples):
    lbl = c12 - 1
    samples_by_class.setdefault(lbl, []).append(i)

ax_idx = 0
for lbl in present_classes:
    if ax_idx >= len(axes_flat):
        break
    indices = samples_by_class.get(lbl, [])
    if not indices:
        continue
    for row_off in range(n_rows):
        if ax_idx >= len(axes_flat):
            break
        sample_idx = indices[min(row_off, len(indices) - 1)]
        tensor, label = display_ds[sample_idx]
        if isinstance(tensor, torch.Tensor):
            img = tensor.permute(1, 2, 0).numpy()
            img = (img * 255).clip(0, 255).astype(np.uint8)
        else:
            img = tensor
        axes_flat[ax_idx].imshow(img)
        axes_flat[ax_idx].set_title(
            f"{CLASS_NAMES_SHORT[label]}\n{CLASS_NAMES[label]}", fontsize=9
        )
        axes_flat[ax_idx].axis("off")
        ax_idx += 1

for i in range(ax_idx, len(axes_flat)):
    axes_flat[i].axis("off")

plt.suptitle("Sample Crops per Present Class (AI Hub 091)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Train the Classifier

Uses the same hyperparameters as `classifier_base.yaml`:
- **Backbone:** EfficientNet-B0 (ImageNet pretrained)
- **Loss:** Focal loss (γ=2.0, label smoothing=0.1, class-weighted)
- **Optimizer:** AdamW (lr=3e-4, weight_decay=1e-4)
- **Scheduler:** Cosine annealing with warm restarts (T₀=10, T_mult=2)
- **Sampling:** Sqrt-smoothed inverse-frequency weighted
- **Early stopping:** patience=10 on val_f1_macro
- **Epochs:** 50 max

In [ ]:
# ── Hyperparameters (from classifier_base.yaml) ────────────────────────

CFG = {
    "model": {
        "backbone": "efficientnet_b0",
        "pretrained": True,
        "input_size": INPUT_SIZE,
        "num_classes": NUM_CLASSES,
        "dropout": 0.3,
    },
    "training": {
        "epochs": 50,
        "batch_size": 64,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "scheduler": "cosine_warm_restarts",
        "t_0": 10,
        "t_mult": 2,
        "early_stopping": {
            "enabled": True,
            "patience": 10,
            "metric": "val_f1_macro",
        },
    },
    "loss": {
        "type": "focal",
        "gamma": 2.0,
        "label_smoothing": 0.1,
        "class_weights": True,
    },
    "sampling": {
        "weighted": True,
        "smoothing": 0.5,
    },
    "data": {
        "workers": 2,  # Colab has limited CPU cores
        "min_crop_px": 16,
    },
}

In [ ]:
# ── Build model, loss, optimizer, scheduler ────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model_cfg = CFG["model"]
train_cfg = CFG["training"]

model = timm.create_model(
    model_cfg["backbone"],
    pretrained=model_cfg["pretrained"],
    num_classes=NUM_CLASSES,
    drop_rate=model_cfg["dropout"],
).to(device)

# Weighted sampler
sampler_factory = ClassWeightedSampler(train_ds, smoothing=CFG["sampling"]["smoothing"])
train_sampler = sampler_factory.build()
class_weights = sampler_factory.class_weights_tensor()

# Loss
criterion = FocalLoss(
    gamma=CFG["loss"]["gamma"],
    alpha=class_weights,
    label_smoothing=CFG["loss"]["label_smoothing"],
).to(device)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg["lr"],
    weight_decay=train_cfg["weight_decay"],
)

# Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=train_cfg["t_0"],
    T_mult=train_cfg["t_mult"],
)

# Data loaders
batch_size = train_cfg["batch_size"]
workers = CFG["data"]["workers"]

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    sampler=train_sampler,
    num_workers=workers,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=workers,
    pin_memory=True,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {model_cfg['backbone']}  |  {total_params/1e6:.1f}M params ({trainable_params/1e6:.1f}M trainable)")
print(f"Batches per epoch: {len(train_loader)} train, {len(val_loader)} val")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    n_samples = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)
    return running_loss / max(n_samples, 1)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    per_class_correct = [0] * NUM_CLASSES
    per_class_total = [0] * NUM_CLASSES
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        for c in range(NUM_CLASSES):
            mask = labels == c
            per_class_correct[c] += (preds[mask] == c).sum().item()
            per_class_total[c] += mask.sum().item()

    per_class_acc = [
        per_class_correct[c] / max(per_class_total[c], 1) for c in range(NUM_CLASSES)
    ]
    macro_f1 = sum(per_class_acc) / NUM_CLASSES

    return {
        "val_loss": running_loss / max(total, 1),
        "val_accuracy": correct / max(total, 1),
        "val_f1_macro": macro_f1,
        "per_class_accuracy": per_class_acc,
        "all_preds": all_preds,
        "all_labels": all_labels,
    }


# ── Run training ──────────────────────────────────────────────────────

SAVE_DIR = Path("/content/runs/classifier")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

epochs = train_cfg["epochs"]
patience = train_cfg["early_stopping"]["patience"]
metric_key = train_cfg["early_stopping"]["metric"]

best_metric = -float("inf")
best_state = None
epochs_without_improvement = 0
history = []

print(f"Training for up to {epochs} epochs (early stopping patience={patience})\n")

for epoch in range(epochs):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = validate(model, val_loader, criterion, device)
    scheduler.step()
    elapsed = time.time() - t0

    current_metric = val_metrics[metric_key]
    lr = optimizer.param_groups[0]["lr"]

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["val_loss"],
        "val_accuracy": val_metrics["val_accuracy"],
        "val_f1_macro": val_metrics["val_f1_macro"],
        "lr": lr,
        "elapsed_s": round(elapsed, 1),
    }
    history.append(record)

    improved = ""
    if current_metric > best_metric:
        best_metric = current_metric
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_metric": best_metric,
                "config": CFG,
            },
            SAVE_DIR / "best.pt",
        )
        improved = " ★ new best"
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch+1:3d}/{epochs}  "
        f"train_loss={train_loss:.4f}  "
        f"val_acc={val_metrics['val_accuracy']:.4f}  "
        f"val_f1={val_metrics['val_f1_macro']:.4f}  "
        f"lr={lr:.2e}  "
        f"({elapsed:.1f}s){improved}"
    )

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch+1} (patience={patience})")
        break

# Save final best checkpoint
if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(
        {"model_state_dict": best_state, "config": CFG, "best_metric": best_metric},
        SAVE_DIR / "best.pt",
    )

(SAVE_DIR / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
print(f"\nTraining complete — best {metric_key}={best_metric:.4f}")
print(f"Checkpoint saved to {SAVE_DIR / 'best.pt'}")

---
## 5. Evaluate — Training Curves, Per-Class Accuracy, Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt

epochs_range = [r["epoch"] + 1 for r in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(epochs_range, [r["train_loss"] for r in history], label="Train loss")
axes[0].plot(epochs_range, [r["val_loss"] for r in history], label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, [r["val_accuracy"] for r in history], label="Val accuracy", color="green")
axes[1].plot(epochs_range, [r["val_f1_macro"] for r in history], label="Val F1 (macro)", color="orange")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("Validation Metrics")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(epochs_range, [r["lr"] for r in history], color="red")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("Learning Rate Schedule")
axes[2].set_yscale("log")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "training_curves.png", dpi=150)
plt.show()

In [ ]:
# Per-class accuracy on validation set
final_metrics = validate(model, val_loader, criterion, device)

fig, ax = plt.subplots(figsize=(12, 5))
per_class = final_metrics["per_class_accuracy"]
colors = ["#2ecc71" if a >= 0.8 else "#f39c12" if a >= 0.5 else "#e74c3c" for a in per_class]
bars = ax.bar(CLASS_NAMES_SHORT, per_class, color=colors)
ax.set_ylabel("Accuracy")
ax.set_title(f"Per-Class Validation Accuracy  (overall={final_metrics['val_accuracy']:.3f}, macro_f1={final_metrics['val_f1_macro']:.3f})")
ax.set_ylim(0, 1.05)
ax.axhline(y=0.8, color="gray", linestyle="--", alpha=0.5, label="80% threshold")
ax.legend()
for bar, v in zip(bars, per_class):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{v:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(SAVE_DIR / "per_class_accuracy.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    final_metrics["all_labels"],
    final_metrics["all_preds"],
    labels=list(range(NUM_CLASSES)),
)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES_SHORT,
)
disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45)
ax.set_title("Confusion Matrix — Validation Set")
plt.tight_layout()
plt.savefig(SAVE_DIR / "confusion_matrix.png", dpi=150)
plt.show()

---
## 6. Export to TFLite

Uses `ai_edge_torch` to convert the PyTorch model directly to TFLite
with a fixed batch size of 1 (suitable for on-device single-image inference).

In [ ]:
import ai_edge_torch

EXPORT_DIR = Path("/content/export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load best checkpoint
ckpt = torch.load(SAVE_DIR / "best.pt", map_location="cpu", weights_only=False)
export_model = timm.create_model(
    CFG["model"]["backbone"],
    pretrained=False,
    num_classes=NUM_CLASSES,
    drop_rate=CFG["model"]["dropout"],
)
export_model.load_state_dict(ckpt["model_state_dict"])
export_model.eval()

sample_input = (torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE),)
edge_model = ai_edge_torch.convert(export_model, sample_input)

tflite_path = EXPORT_DIR / "classifier.tflite"
edge_model.export(str(tflite_path))

size_mb = tflite_path.stat().st_size / 1e6
print(f"TFLite model exported to {tflite_path} ({size_mb:.1f} MB)")

# Write metadata alongside the model
from datetime import datetime, timezone

metadata = {
    "model_type": "classifier",
    "architecture": CFG["model"]["backbone"],
    "input_size": INPUT_SIZE,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "best_metric": best_metric,
    "exported_at": datetime.now(timezone.utc).isoformat(),
}
(EXPORT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
print("Metadata written.")

---
## 7. Download

Download `classifier.tflite` and place it at:
```
apps/mobile_flutter/assets/models/classifier.tflite
```

The Flutter app will automatically pick it up — no code changes needed.

In [ ]:
from google.colab import files

files.download(str(tflite_path))
print("\nDone! Place classifier.tflite in apps/mobile_flutter/assets/models/")